# Z-Image-Turbo GGUF · Kaggle 2×T4

`stable-diffusion.cpp` · Q4_K · 双 T4 常驻 Worker · AES-GCM 加密回传


## 1. 配置


In [8]:
import os, sys, time, uuid, queue, base64, hashlib, shutil, threading, asyncio, subprocess
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "aiohttp", "cryptography", "requests"
], check=True)

import requests
import aiohttp
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

BASE = os.getenv("BASE_URL", "https://ranran-sana.202820.xyz").rstrip("/")
TOKEN = os.getenv("KAGGLE_HUB_TOKEN") or os.getenv("PASSWORD") or "wangran"

MODEL = "z-image-turbo-gguf"
if "WORKER_ID" not in globals():
    WORKER_ID = f"zimage-{uuid.uuid4().hex[:8]}"

PORTS = [12340, 12341]
STEPS = 8
CFG_SCALE = 1.0
IO_WORKERS = 2
UPLOAD_QUEUE_SIZE = 2

CLAIM_URL = f"{BASE}/task/next"
UPLOAD_URL = f"{BASE}/upload"
REGISTER_URL = f"{BASE}/worker/register"
HEARTBEAT_URL = f"{BASE}/worker/heartbeat"
FAIL_URL = f"{BASE}/task/fail"

gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True
).strip().splitlines()

assert len(gpu_names) >= 2, f"需要 2 张 GPU，当前只有 {len(gpu_names)} 张"
print("worker:", WORKER_ID)
print("gpu:", gpu_names[:2])


worker: zimage-f849f191
gpu: ['Tesla T4', 'Tesla T4']


## 2. stable-diffusion.cpp


In [9]:
TAG = "sdcpp-t4-sm75-de298c225bed"
ASSET = "stable-diffusion-cpp-linux-cuda-t4-sm75-de298c225bed.tar.gz"
SHA256 = "00075e290f3384ab08622ad4070476e9a4b4283bb8eb128d1e8248271106d30b"

WORK = Path("/kaggle/working/zimage_gguf")
ARCHIVE = WORK / ASSET
PREBUILT = WORK / TAG
WORK.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not ARCHIVE.exists() or sha256_file(ARCHIVE) != SHA256:
    ARCHIVE.unlink(missing_ok=True)
    subprocess.run([
        "curl", "-L", "--fail", "--retry", "3", "--retry-delay", "2",
        "-o", str(ARCHIVE),
        f"https://github.com/xiaoqianran/kaggle-build/releases/download/{TAG}/{ASSET}"
    ], check=True)

assert sha256_file(ARCHIVE) == SHA256, "stable-diffusion.cpp 预编译包 SHA256 不匹配"

candidates = list(PREBUILT.rglob("sd-server")) if PREBUILT.exists() else []
if not candidates:
    shutil.rmtree(PREBUILT, ignore_errors=True)
    PREBUILT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", str(PREBUILT)], check=True)
    candidates = list(PREBUILT.rglob("sd-server"))

assert candidates, "预编译包中未找到 sd-server"
SD_SERVER = str(min(candidates, key=lambda p: len(str(p))))
os.chmod(SD_SERVER, os.stat(SD_SERVER).st_mode | 0o111)

lib_dirs = sorted({str(p.parent) for p in PREBUILT.rglob("*.so*") if p.is_file()})
if lib_dirs:
    old = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + ([old] if old else []))

print("sd-server:", SD_SERVER)


sd-server: /kaggle/working/zimage_gguf/sdcpp-t4-sm75-de298c225bed/bin/sd-server


## 3. 模型


In [10]:
from huggingface_hub import hf_hub_download

MODEL_DIR = WORK / "models"
MODEL_DIR.mkdir(exist_ok=True)

DIFFUSION = hf_hub_download(
    "leejet/Z-Image-Turbo-GGUF",
    "z_image_turbo-Q4_K.gguf",
    local_dir=MODEL_DIR,
)
LLM = hf_hub_download(
    "unsloth/Qwen3-4B-Instruct-2507-GGUF",
    "Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
    local_dir=MODEL_DIR,
)
VAE = hf_hub_download(
    "Comfy-Org/z_image_turbo",
    "split_files/vae/ae.safetensors",
    local_dir=MODEL_DIR,
)

for path in (DIFFUSION, LLM, VAE):
    assert Path(path).is_file() and Path(path).stat().st_size > 0, path

print("models ready")


models ready


## 4. 启动双 T4 server


In [11]:
def server_ready(port):
    try:
        return requests.get(
            f"http://127.0.0.1:{port}/v1/models",
            timeout=2
        ).ok
    except requests.RequestException:
        return False

def stop_servers():
    for p in globals().get("SD_PROCS", []):
        if p.poll() is None:
            p.terminate()
    for p in globals().get("SD_PROCS", []):
        if p.poll() is None:
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
                p.wait(timeout=5)
    for f in globals().get("SD_LOGS", []):
        try:
            f.close()
        except Exception:
            pass

def wait_server(port, proc, log_path, timeout=600):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            tail = Path(log_path).read_text(errors="ignore")[-8000:]
            raise RuntimeError(f"sd-server :{port} exited\n{tail}")
        if server_ready(port):
            return
        time.sleep(1)
    raise TimeoutError(f"sd-server :{port} 启动超时")

existing = globals().get("SD_PROCS", [])
reuse = (
    len(existing) == 2
    and all(p.poll() is None for p in existing)
    and all(server_ready(p) for p in PORTS)
)

if reuse:
    print("双 T4 server 已在运行，保持现有进程")
else:
    stop_servers()
    SD_PROCS, SD_LOGS = [], []

    for gpu, port in enumerate(PORTS):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)

        log_path = f"/kaggle/working/zimage_gpu{gpu}.log"
        log_file = open(log_path, "w", buffering=1)
        cmd = [
            SD_SERVER,
            "--diffusion-model", DIFFUSION,
            "--vae", VAE,
            "--llm", LLM,
            "--backend", "cuda0",
            "--diffusion-fa",
            "--cfg-scale", str(CFG_SCALE),
            "--listen-ip", "127.0.0.1",
            "--listen-port", str(port),
        ]

        proc = subprocess.Popen(
            cmd,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
        SD_PROCS.append(proc)
        SD_LOGS.append(log_file)
        wait_server(port, proc, log_path)
        print(f"GPU{gpu} ready :{port}")


双 T4 server 已在运行，保持现有进程


## 5. 推理与回传编码自检


In [12]:
FFMPEG = shutil.which("ffmpeg")

if not FFMPEG:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"
    ], check=True)
    import imageio_ffmpeg
    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

def png_to_webp(png_bytes):
    import tempfile

    fd, out_path = tempfile.mkstemp(suffix=".webp")
    os.close(fd)

    try:
        p = subprocess.run([
            FFMPEG,
            "-hide_banner", "-loglevel", "error",
            "-i", "pipe:0",
            "-frames:v", "1",
            "-c:v", "libwebp",
            "-quality", "90",
            "-y", out_path,
        ], input=png_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        if p.returncode != 0:
            raise RuntimeError(
                "WebP encode failed: " + p.stderr.decode(errors="ignore")[-1000:]
            )

        webp = Path(out_path).read_bytes()
    finally:
        Path(out_path).unlink(missing_ok=True)

    if len(webp) < 20 or webp[:4] != b"RIFF" or webp[8:12] != b"WEBP":
        raise RuntimeError("ffmpeg 返回的不是有效 WebP")

    riff_size = int.from_bytes(webp[4:8], "little")
    if riff_size != len(webp) - 8:
        raise RuntimeError(
            f"WebP RIFF size invalid: header={riff_size}, actual={len(webp)-8}"
        )

    return webp

KEY = hashlib.sha256(TOKEN.encode()).digest()

def encrypt_blob(data):
    nonce = os.urandom(12)
    return nonce + AESGCM(KEY).encrypt(nonce, data, None)

def generate(session, port, task):
    payload = {
        "prompt": task["prompt"],
        "negative_prompt": "",
        "width": int(task.get("width", 1024)),
        "height": int(task.get("height", 1024)),
        "steps": int(task.get("steps", STEPS)),
        "cfg_scale": CFG_SCALE,
        "seed": int(task["seed"]),
        "batch_size": 1,
    }

    r = session.post(
        f"http://127.0.0.1:{port}/sdapi/v1/txt2img",
        json=payload,
        timeout=(10, 900),
    )
    if r.status_code >= 400:
        raise RuntimeError(f"sd-server HTTP {r.status_code}: {r.text[:1000]}")

    data = r.json()
    images = data.get("images") or []
    if not images:
        raise RuntimeError(f"sd-server 返回中没有 images: {str(data)[:1000]}")

    png = base64.b64decode(images[0])
    if not png.startswith(b"\x89PNG\r\n\x1a\n"):
        raise RuntimeError("sd-server 返回的 image 不是 PNG")

    return png, payload["steps"]

warm_task = {
    "prompt": "a simple red cube on a neutral background",
    "width": 256,
    "height": 256,
    "steps": 1,
    "seed": 12345,
}

with requests.Session() as s:
    png, _ = generate(s, PORTS[0], warm_task)

webp = png_to_webp(png)
encrypted = encrypt_blob(webp)
nonce, ciphertext = encrypted[:12], encrypted[12:]
assert AESGCM(KEY).decrypt(nonce, ciphertext, None) == webp

print(f"runtime OK · PNG {len(png)/1024:.1f} KiB → WebP {len(webp)/1024:.1f} KiB → AES-GCM OK")


runtime OK · PNG 94.2 KiB → WebP 4.0 KiB → AES-GCM OK


## 6. Remote Dispatcher


In [13]:
class Dispatcher:
    def __init__(self):
        self.stop_event = threading.Event()
        self.uploads = queue.Queue(maxsize=UPLOAD_QUEUE_SIZE)
        self.network_thread = None
        self.gpu_threads = []

    def is_alive(self):
        threads = [self.network_thread, *self.gpu_threads]
        return any(t is not None and t.is_alive() for t in threads)

    def auth_headers(self):
        return {"Authorization": f"Bearer {TOKEN}"}

    def report_fail(self, task_id, error, requeue=True):
        try:
            r = requests.post(
                FAIL_URL,
                headers=self.auth_headers(),
                json={
                    "id": int(task_id),
                    "error": str(error)[:2000],
                    "requeue": bool(requeue),
                },
                timeout=15,
            )
            if r.status_code >= 400 and r.status_code != 404:
                print(f"! fail report #{task_id}: HTTP {r.status_code} {r.text[:300]}")
        except Exception as e:
            print(f"! fail report #{task_id}: {e}")

    async def register(self, session):
        payload = {
            "worker_id": WORKER_ID,
            "model": MODEL,
            "gpus": gpu_names[:2],
            "runtime": "stable-diffusion.cpp",
            "concurrency": 2,
            "meta": {"ports": PORTS, "cfg_scale": CFG_SCALE},
        }
        async with session.post(REGISTER_URL, json=payload) as r:
            if r.status >= 400:
                raise RuntimeError(f"register HTTP {r.status}: {await r.text()}")
        print(f"✓ registered {WORKER_ID} -> {MODEL}")

    async def heartbeat_loop(self, session):
        while not self.stop_event.is_set():
            try:
                payload = {
                    "worker_id": WORKER_ID,
                    "local_queue": 0,
                    "upload_queue": self.uploads.qsize(),
                    "meta": {
                        "gpu_threads": sum(t.is_alive() for t in self.gpu_threads),
                        "servers": [
                            p.poll() is None for p in globals().get("SD_PROCS", [])
                        ],
                    },
                }
                async with session.post(HEARTBEAT_URL, json=payload) as r:
                    if r.status == 404:
                        await self.register(session)
                    elif r.status >= 400:
                        raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
            except Exception as e:
                if not self.stop_event.is_set():
                    print(f"! heartbeat: {e}")
            await asyncio.sleep(10)

    async def upload_one(self, session, item):
        try:
            webp = await asyncio.to_thread(png_to_webp, item["png"])
            encrypted = await asyncio.to_thread(encrypt_blob, webp)
        except Exception as e:
            print(f'! encode #{item["id"]}: {e}')
            await asyncio.to_thread(
                self.report_fail, item["id"], f"encode: {e}", False
            )
            return

        for attempt in range(8):
            try:
                form = aiohttp.FormData()
                form.add_field(
                    "file",
                    encrypted,
                    filename=f'{item["id"]:06d}.bin',
                    content_type="application/octet-stream",
                )
                for k in (
                    "id", "model", "worker_id", "gpu",
                    "seed", "prompt", "seconds", "steps"
                ):
                    form.add_field(k, str(item[k]))

                async with session.post(UPLOAD_URL, data=form) as r:
                    text = await r.text()
                    if 200 <= r.status < 300:
                        print(f'↑ #{item["id"]:03d} | GPU{item["gpu"]} → PC')
                        return
                    if r.status == 409:
                        print(f'! stale #{item["id"]}: {text[:300]}')
                        return
                    if r.status in (400, 401, 403):
                        raise RuntimeError(f"permanent HTTP {r.status}: {text[:500]}")
                    raise RuntimeError(f"HTTP {r.status}: {text[:500]}")

            except Exception as e:
                if attempt == 7:
                    print(f'! upload #{item["id"]}: {e}')
                    await asyncio.to_thread(
                        self.report_fail, item["id"], f"upload: {e}", True
                    )
                    return
                await asyncio.sleep(min(30, 2 ** attempt))

    async def upload_loop(self, session):
        while not self.stop_event.is_set() or not self.uploads.empty():
            try:
                item = await asyncio.to_thread(self.uploads.get, True, 1)
            except queue.Empty:
                continue
            await self.upload_one(session, item)

    async def network_main(self):
        timeout = aiohttp.ClientTimeout(total=120, connect=10)
        async with aiohttp.ClientSession(
            headers=self.auth_headers(),
            timeout=timeout,
        ) as session:
            await self.register(session)
            await asyncio.gather(
                self.heartbeat_loop(session),
                *(self.upload_loop(session) for _ in range(IO_WORKERS)),
            )

    def run_network(self):
        try:
            asyncio.run(self.network_main())
        except Exception as e:
            if not self.stop_event.is_set():
                print(f"! network thread: {e}")
                self.stop_event.set()

    def claim_task(self, session):
        while not self.stop_event.is_set():
            try:
                r = session.get(
                    CLAIM_URL,
                    params={"model": MODEL, "worker_id": WORKER_ID},
                    timeout=(10, 35),
                )
                if r.status_code == 204:
                    continue
                if r.status_code == 401:
                    print("! claim unauthorized，停止 Dispatcher")
                    self.stop_event.set()
                    return None
                if r.status_code >= 400:
                    print(f"! claim HTTP {r.status_code}: {r.text[:300]}")
                    time.sleep(2)
                    continue
                return r.json()
            except requests.RequestException as e:
                if not self.stop_event.is_set():
                    print(f"! claim: {e}")
                    time.sleep(2)
        return None

    def gpu_loop(self, gpu, port):
        hub = requests.Session()
        hub.headers.update(self.auth_headers())
        sd = requests.Session()

        try:
            while not self.stop_event.is_set():
                task = self.claim_task(hub)
                if task is None:
                    continue

                print(f'↓ #{task["id"]:03d} | GPU{gpu} | {task["prompt"][:70]}')
                start = time.perf_counter()

                try:
                    last = None
                    for attempt in range(2):
                        try:
                            png, steps = generate(sd, port, task)
                            last = None
                            break
                        except Exception as e:
                            last = e
                            if attempt == 0:
                                print(f'↻ #{task["id"]:03d} | GPU{gpu} retry | {e}')
                                time.sleep(1)

                    if last is not None:
                        raise last

                    seconds = round(time.perf_counter() - start, 3)
                    item = {
                        "id": task["id"],
                        "model": MODEL,
                        "worker_id": WORKER_ID,
                        "gpu": gpu,
                        "seed": task["seed"],
                        "prompt": task["prompt"],
                        "seconds": seconds,
                        "steps": steps,
                        "png": png,
                    }

                    while True:
                        try:
                            self.uploads.put(item, timeout=1)
                            break
                        except queue.Full:
                            if self.stop_event.is_set():
                                continue

                    print(
                        f'✓ #{task["id"]:03d} | GPU{gpu} | '
                        f'{task.get("width",1024)}x{task.get("height",1024)} | '
                        f'{steps} steps | {seconds:.2f}s'
                    )

                except Exception as e:
                    print(f'✗ #{task.get("id")} | GPU{gpu} | {e}')
                    self.report_fail(task["id"], e, True)

        finally:
            hub.close()
            sd.close()

    def start(self):
        self.network_thread = threading.Thread(
            target=self.run_network,
            name="zimage-network",
            daemon=True,
        )
        self.gpu_threads = [
            threading.Thread(
                target=self.gpu_loop,
                args=(gpu, port),
                name=f"zimage-gpu{gpu}",
                daemon=True,
            )
            for gpu, port in enumerate(PORTS)
        ]

        self.network_thread.start()
        for t in self.gpu_threads:
            t.start()

        print(
            f"Dispatcher ready · {WORKER_ID} · "
            f"GPU0:{PORTS[0]} + GPU1:{PORTS[1]} · "
            f"Q4_K · {STEPS} steps"
        )
        return self

    def stop(self):
        self.stop_event.set()
        print("Dispatcher stop requested；sd-server 保持运行")

    def status(self):
        print("worker:", WORKER_ID)
        print("stop:", self.stop_event.is_set())
        print("upload queue:", self.uploads.qsize(), "/", self.uploads.maxsize)
        print("network:", bool(self.network_thread and self.network_thread.is_alive()))
        for i, t in enumerate(self.gpu_threads):
            print(f"gpu{i} thread:", t.is_alive())
        for i, p in enumerate(globals().get("SD_PROCS", [])):
            print(f"sd-server gpu{i}:", "running" if p.poll() is None else f"dead({p.returncode})")

old = globals().get("DISPATCHER")
if old is not None and old.is_alive():
    print("Dispatcher 已在运行，不重复启动")
else:
    DISPATCHER = Dispatcher().start()


Dispatcher 已在运行，不重复启动


## 7. 状态 / 可选停止


In [14]:
DISPATCHER.status()

# 仅停止远程任务领取与上传线程：
# DISPATCHER.stop()

# 真正释放 GPU 时再执行：
# DISPATCHER.stop()
# stop_servers()


worker: zimage-f849f191
stop: False
upload queue: 0 / 2
network: True
gpu0 thread: True
gpu1 thread: True
sd-server gpu0: running
sd-server gpu1: running
↓ #093 | GPU0 | - Low angle tracking shot, giant, fluffy dandelion seeds floating like
↓ #092 | GPU1 | - Cinematic macro wide shot, a single, perfect morning dewdrop hanging
✓ #093 | GPU0 | 1024x1024 | 8 steps | 42.68s
↓ #094 | GPU0 | - Extreme close-up, a luminous, iridescent beetle resting on the key o
✓ #092 | GPU1 | 1024x1024 | 8 steps | 45.30s
↓ #095 | GPU1 | - Medium shot, an open, ancient leather-bound book lying on a wooden d
↑ #092 | GPU1 → PC
↑ #093 | GPU0 → PC
✓ #095 | GPU1 | 1024x1024 | 8 steps | 41.98s
↓ #096 | GPU1 | - Aerial panoramic shot, a backyard lawn at twilight transformed into 
✓ #094 | GPU0 | 1024x1024 | 8 steps | 44.77s
↓ #097 | GPU0 | - Cinematic wide establishing shot, an ancient, moss-covered wooden te
↑ #095 | GPU1 → PC
↑ #094 | GPU0 → PC
